# Entrega 1 — Análise Comparativa de Representações

**Ouvidoria Inteligente: Triagem Semântica de Manifestações Cidadãs**

A triagem atual da ouvidoria busca por palavras-chave, e por isso “asfalto esburacado” e “rua com buraco” são tratados como problemas diferentes. Este notebook compara três formas de representar as 40 manifestações:

- **BoW:** contagem de palavras, representação esparsa.
- **TF-IDF:** pesos por raridade, também esparsa.
- **Embeddings:** vetores densos de um modelo multilíngue.

A comparação mede a similaridade de cosseno nos três pares pedidos no enunciado.

> **Dados:** o `manifestacoes.json` é produzido pelo próprio aluno. Este conjunto de 40 manifestações segue o esquema da seção 2 do enunciado (id, data, categoria_oficial, texto), e os textos dos pares contêm os trechos citados no enunciado.

In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from ouvidoria import MODELO_PADRAO, STOPWORDS_PT, carregar_manifestacoes, carregar_modelo, gerar_embeddings

df = pd.DataFrame(carregar_manifestacoes())
print(f"{len(df)} manifestações, campos: {list(df.columns)}")
df["categoria_oficial"].value_counts()

40 manifestações, campos: ['id', 'data', 'categoria_oficial', 'texto']


categoria_oficial
infraestrutura    9
saúde             8
segurança         8
educação          8
meio ambiente     7
Name: count, dtype: int64

In [2]:
df.head(8)

,id,data,categoria_oficial,texto
0,M001,2026-08-01,infraestrutura,A calçada da Rua das Acácias está toda quebrad...
1,M002,2026-08-01,saúde,Fui ao pronto atendimento do Jardim Esperança ...
2,M003,2026-08-02,infraestrutura,"Tem um buraco enorme na Av. Brasil, perto do c..."
3,M004,2026-08-02,segurança,Assaltos frequentes no ponto de ônibus da Rua ...
4,M005,2026-08-03,infraestrutura,"Toda vez que chove forte, a Rua do Canal, no b..."
5,M006,2026-08-04,educação,A Escola Municipal Monteiro Lobato está sem me...
6,M007,2026-08-04,meio ambiente,O terreno baldio da Rua Ipê Amarelo virou depó...
7,M008,2026-08-05,saúde,Posto de saúde sem médico: a unidade do bairro...


## 1) Três representações do corpus

- **BoW** (`CountVectorizer`) e **TF-IDF** (`TfidfVectorizer`) usam uma lista de stopwords em português. O scikit-learn só traz a lista em inglês.
- **Embeddings:** `paraphrase-multilingual-MiniLM-L12-v2`, modelo multilíngue sugerido no enunciado. Os vetores são normalizados (norma 1).

In [3]:
textos = df["texto"].tolist()

bow = CountVectorizer(stop_words=STOPWORDS_PT)
X_bow = bow.fit_transform(textos)

tfidf = TfidfVectorizer(stop_words=STOPWORDS_PT)
X_tfidf = tfidf.fit_transform(textos)

modelo = carregar_modelo(MODELO_PADRAO)
X_emb = gerar_embeddings(textos, modelo)

pd.DataFrame({
    "Representação": ["BoW", "TF-IDF", "Embeddings"],
    "Dimensões": [X_bow.shape[1], X_tfidf.shape[1], X_emb.shape[1]],
    "Tipo": ["esparsa (contagens)", "esparsa (pesos TF-IDF)", "densa"],
    "Valores não nulos": [
        f"{X_bow.nnz / np.prod(X_bow.shape):.1%}",
        f"{X_tfidf.nnz / np.prod(X_tfidf.shape):.1%}",
        f"{np.count_nonzero(X_emb) / X_emb.size:.1%}",
    ],
})

,Representação,Dimensões,Tipo,Valores não nulos
0,BoW,561,esparsa (contagens),3.6%
1,TF-IDF,561,esparsa (pesos TF-IDF),3.6%
2,Embeddings,384,densa,100.0%


## 2) Similaridade de cosseno nos três pares

In [4]:
PARES = [
    ("M003", "M017", "buraco na Av. Brasil × asfalto esburacado da avenida principal", "mesmo problema"),
    ("M008", "M022", "posto de saúde sem médico × falta atendimento no PSF", "mesmo problema"),
    ("M008", "M031", "posto de saúde sem médico × lâmpada queimada na praça", "temas diferentes"),
]
pos = {mid: i for i, mid in enumerate(df["id"])}


def cosseno_esparso(X, a, b):
    return float(cosine_similarity(X[pos[a]], X[pos[b]])[0, 0])


tabela = pd.DataFrame([
    {
        "Par": f"{a} × {b}",
        "Descrição": descricao,
        "Relação": relacao,
        "BoW": cosseno_esparso(X_bow, a, b),
        "TF-IDF": cosseno_esparso(X_tfidf, a, b),
        "Embeddings": float(X_emb[pos[a]] @ X_emb[pos[b]]),
    }
    for a, b, descricao, relacao in PARES
]).set_index("Par")
tabela.round(3)

,Descrição,Relação,BoW,TF-IDF,Embeddings
Par,,,,,
M003 × M017,buraco na Av. Brasil × asfalto esburacado da a...,mesmo problema,0.143,0.133,0.715
M008 × M022,posto de saúde sem médico × falta atendimento ...,mesmo problema,0.167,0.159,0.848
M008 × M031,posto de saúde sem médico × lâmpada queimada n...,temas diferentes,0.000,0.000,0.225


In [5]:
analisar = bow.build_analyzer()
for a, b, *_ in PARES:
    comuns = sorted(set(analisar(textos[pos[a]])) & set(analisar(textos[pos[b]])))
    print(f"{a}: {textos[pos[a]]}")
    print(f"{b}: {textos[pos[b]]}")
    print(f"→ termos em comum no BoW/TF-IDF: {comuns or 'nenhum'}\n")

M003: Tem um buraco enorme na Av. Brasil, perto do cruzamento com a Rua Sete de Setembro. Os carros desviam pela contramão e já houve acidente com moto.
M017: O asfalto todo esburacado da avenida principal, na altura da Sete de Setembro, já derrubou um motociclista. A via precisa de recapeamento urgente.
→ termos em comum no BoW/TF-IDF: ['sete', 'setembro']

M008: Posto de saúde sem médico: a unidade do bairro São José está sem clínico geral há mais de um mês. A gente chega de madrugada para pegar ficha e volta para casa sem consulta.
M022: Falta atendimento no PSF do São José. Desde o mês passado não aparece doutor, a recepcionista manda voltar outro dia e ninguém é atendido.
→ termos em comum no BoW/TF-IDF: ['josé', 'mês', 'são']

M008: Posto de saúde sem médico: a unidade do bairro São José está sem clínico geral há mais de um mês. A gente chega de madrugada para pegar ficha e volta para casa sem consulta.
M031: Lâmpada queimada na praça Dom Adauto: a praça está no escuro há semanas

## 3) Discussão

- **M003 × M017 (o mesmo buraco):** BoW ≈ 0,14 e TF-IDF ≈ 0,13, e essa similaridade vem só de “sete” e “setembro”, isto é, do nome da rua e não do problema. Para os modelos esparsos, “buraco” e “esburacado”, e “Av.” e “avenida”, são tokens diferentes. Os embeddings dão ≈ 0,72: o modelo reconhece que os dois textos falam de asfalto danificado na mesma avenida.
- **M008 × M022 (o mesmo problema no posto):** de novo o esparso só casa “são”, “josé” e “mês” (bairro e tempo), com BoW ≈ 0,17 e TF-IDF ≈ 0,16. “Sem médico” e “não aparece doutor” não têm nenhuma palavra em comum. Os embeddings dão ≈ 0,85, o maior dos três pares.
- **M008 × M031 (temas diferentes):** vale 0 nas duas representações esparsas, porque não há nenhum termo em comum, e ≈ 0,23 nos embeddings, bem abaixo dos pares do mesmo problema.
- **Separação entre os casos:** nas representações esparsas, os pares duplicados ficam quase tão distantes quanto o par sem relação (0,13–0,17 contra 0), e a pouca diferença vem de nomes próprios coincidentes. Nos embeddings a separação é grande (0,72–0,85 contra 0,23), que é o que a triagem de duplicatas precisa.

## 4) Conclusão: limitações de cada abordagem

- **BoW** conta palavras. É simples e interpretável, mas não reconhece sinônimos nem variações (“buraco” e “esburacado”), e a similaridade pode vir de palavras irrelevantes para o problema, como nome de rua ou de bairro.
- **TF-IDF** reduz o peso de termos comuns, mas herda a limitação central do BoW: continua sendo casamento exato de termos. A representação é esparsa (mais de 96% de zeros) e não tem noção de significado; em textos curtos, poucas palavras coincidentes decidem tudo.
- **Embeddings** capturam significado e paráfrases, justamente o que falta às outras duas. Em troca, são opacos (as 384 dimensões não têm interpretação), dependem do modelo escolhido (idioma e domínio), custam mais para calcular e diluem identificadores exatos, como número de protocolo ou nome de rua.

Para a ouvidoria, os embeddings resolvem o problema das duplicatas não detectadas. A busca por palavra-chave ainda serve para filtros exatos, como bairro, rua ou protocolo.